# Day 5: Cross-Validation and Evaluation Metrics from Scratch

**Goal:** Implement K-Fold Cross-Validation manually in NumPy, then build
the core evaluation metrics used in classification.

In [33]:
import numpy as np
import pandas as pd
from wrapped_models.decision_tree import DecisionTree
from wrapped_models.xgboost_classifier import XGBoostClassifier
from wrapped_models.random_forest import RandomForest
from wrapped_models.logistic_regression import LogisticRegression

## K-Fold Cross-Validation

A single train/validation split is fragile — depending on which samples
land in validation, your accuracy estimate can be too optimistic or too
pessimistic. K-Fold fixes this by repeating the evaluation K times.

The dataset is divided into K equal **folds**. In each iteration one fold
becomes the validation set and the remaining K-1 folds are used for
training. This repeats until every fold has been the validation set
exactly once.

The final score is the **average metric across all K folds**:

$$\text{CV Score} = \frac{1}{K} \sum_{k=1}^{K} \text{metric}(y_k,\ \hat{y}_k)$$

For example with K=5 and 100 samples:

| Fold | Train indices | Validation indices |
|------|--------------|-------------------|
| 1    | 20–99        | 0–19              |
| 2    | 0–19, 40–99  | 20–39             |
| 3    | 0–39, 60–99  | 40–59             |
| 4    | 0–59, 80–99  | 60–79             |
| 5    | 0–79         | 80–99             |

Two things K-Fold gives you that a single split cannot:
- **Lower variance** — the score is averaged over K different validation
  sets, so one unlucky split cannot skew the result
- **Full data usage** — every sample is used for both training and
  validation across the K rounds

In [34]:
def kfold_split(n: int, k: int, shuffle: bool = True) -> list[tuple[list[int], list[int]]]:
    indices = list(range(n))
    if shuffle:
        np.random.shuffle(indices)

    fold_size = n // k
    remainder = n % k   

    begin = 0
    splits = []

    for _ in range(k):
        end = begin + fold_size if remainder <= 0 else begin + fold_size + 1
        remainder -= 1

        validation_indices = indices[begin : end]
        training_indices = indices[:begin] + indices[end:]
        
        splits.append((training_indices, validation_indices))
        begin = end
    
    return splits

## Executing Cross-Validation

With the split logic in place, we need a function to manage the training and validation process across the folds. This function will take a model, the dataset, and the number of folds, then return the accuracy for each fold.

In [35]:
def cross_validate(model, X: pd.DataFrame, y: np.ndarray, k: int, shuffle: bool) -> float:
    splits = kfold_split(n=len(X), k=k, shuffle=shuffle)
    accuracies = []

    for training_indices, validation_indices in splits:
        # train model on train indices and validate on val ones
        model.fit(X.iloc[training_indices], y[training_indices])
        predictions = np.array(model.predict(X.iloc[validation_indices])) == y[validation_indices]
        
        accuracy = np.mean(predictions)
        accuracies.append(accuracy)

    # ret CV score which is the mean across accuracies
    return float(np.mean(accuracies))

## Testing the Pipeline

We will verify our K-Fold implementation by running it on a dummy dataset using one of our previously built models.

In [36]:
data = {
    'Age': [22, 25, 47, 35, 14, 50, 28, 19, 60, 38],
    'Sex': ['male', 'female', 'female', 'male', 'male', 'female', 'male', 'female', 'male', 'female'],
    'Survived': [0, 1, 1, 0, 1, 1, 0, 1, 0, 1]
}
df_dummy = pd.DataFrame(data)

# logistic regression requires numerical values
df_dummy['Sex'] = df_dummy['Sex'].map({'male': 1, 'female': 0})

X_dummy = df_dummy[['Age', 'Sex']].astype(float)
y_dummy = df_dummy['Survived'].to_numpy()

# initialize all models
models = {
    "Logistic Regression": LogisticRegression(learning_rate=0.01, epochs=100),
    "Decision Tree": DecisionTree(max_depth=3, min_samples=2, random_subspace=False),
    "Random Forest": RandomForest(n_trees=5),
    "XGBoost Classifier": XGBoostClassifier(n_estimators=5, learning_rate=0.3, max_depth=2)
}

# perform K-Fold CV (k=3) for each model
results = {}
for name, model in models.items():
    acc = cross_validate(
        model=model,
        X=X_dummy,
        y=y_dummy,
        k=3,
        shuffle=True
    )
    results[name] = acc

# print results
print("--- K-Fold Cross-Validation Accuracy ---")
for name, acc in results.items():
    print(f"{name:20s} : {acc:.4f}")

--- K-Fold Cross-Validation Accuracy ---
Logistic Regression  : 0.5000
Decision Tree        : 0.6389
Random Forest        : 0.6111
XGBoost Classifier   : 0.8889
